# Лабораторная работа: Рекомендательные системы

## Теоретическая часть

### 1. Суть задачи рекомендательных систем
Рекомендательные системы – это алгоритмы, которые анализируют поведение пользователей и предлагают персонализированные рекомендации товаров, фильмов, музыки и других объектов. Основная цель – предсказать предпочтения пользователей на основе имеющихся данных о взаимодействиях.


### 2. Метод коллаборативной фильтрации
Коллаборативная фильтрация (Collaborative Filtering, CF) – это метод рекомендаций, основанный на анализе поведения пользователей. Он работает на основе предположения, что пользователи с похожими предпочтениями в прошлом будут делать схожий выбор в будущем.

Существует два основных подхода:
1. **User-based CF** – рекомендации строятся на основе сходства пользователей.
2. **Item-based CF** – рекомендации строятся на основе сходства объектов.

### 3. Латентные факторные модели (Matrix Factorization)
Коллаборативная фильтрация может быть реализована через матричное разложение. Пусть у нас есть матрица взаимодействий пользователей и объектов R, где $( R_{u,i} )$ – оценка пользователя ( u ) для объекта ( i ). Тогда разложение можно представить в виде:
$$
R \approx U \cdot V^T
$$
где:
- ( U ) – матрица эмбеддингов пользователей,
- ( V ) – матрица эмбеддингов объектов.

Предсказание рейтинга рассчитывается как:
$$
\hat{R}_{u,i} = U_u \cdot V_i^T
$$

В данной лабораторной работе предполагается использование **нейросетевого метода**, который обучает эмбеддинги пользователей и объектов с помощью полносвязных слоев. Входные данные – индексы пользователей и объектов, которые преобразуются в векторные представления, а затем подаются на вход нейросети.


## Практическая часть
В данной работе вам предлагается реализовать рекомендательную систему на основе метода коллаборативной фильтрации, используя нейросетевую модель. Вы должны:
1. Подготовить данные: загрузить свой датасет (например, рейтинг фильмов, товаров, книг и т. д.).
2. Разбить данные на тренировочный и тестовый наборы.
3. Обучить модель, используя эмбеддинги пользователей и объектов.
4. Оценить качество модели на тестовом наборе.
5. Вывести список рекомендаций для выбранного пользователя.

In [63]:
# Импорты

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import math
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from torch.utils.data import DataLoader, Dataset, random_split

# Определяем устройство (используем GPU, если доступно)
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [64]:
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_rows = 50000

# Читаем файл, но пока без заголовков
df = pd.read_csv('songsDataset.csv', sep=',', header=None, nrows=n_rows)

print(f"Загружено {len(df)} строк (первые {n_rows} из файла).")

# Первая строка - это заголовки
headers = df.iloc[0].values
print(f"Сырые заголовки: {headers}")

# Очищаем заголовки от кавычек и пробелов
clean_headers = [str(h).strip().strip("'") for h in headers]
print(f"Очищенные заголовки: {clean_headers}")

# Устанавливаем очищенные заголовки
df.columns = clean_headers

# Удаляем первую строку (которая была заголовком)
df = df.iloc[1:].reset_index(drop=True)

print(f"\nНазвания колонок после обработки: {list(df.columns)}")

print("\nПервые 5 строк данных:")
print(df.head())

print("\nИнформация о данных:")
print(df.info())

# Преобразуем данные в числовой формат
df['userID'] = pd.to_numeric(df['userID'])
df['songID'] = pd.to_numeric(df['songID'])
df['rating'] = pd.to_numeric(df['rating'])

print("\nСтатистика по рейтингам:")
print(df['rating'].describe())

print("\nКоличество уникальных пользователей:", df['userID'].nunique())
print("Количество уникальных песен:", df['songID'].nunique())

# Проверка пропусков
print("\nПропуски в каждом столбце:")
print(df.isnull().sum())

# Проверяем типы данных
print("\nТипы данных:")
print(df.dtypes)

# Покажем несколько примеров для проверки
print("\nПримеры данных:")
print(df[['userID', 'songID', 'rating']].head(10))

Загружено 50000 строк (первые 50000 из файла).
Сырые заголовки: <StringArray>
[''userID'', ''songID'', ''rating'']
Length: 3, dtype: str
Очищенные заголовки: ['userID', 'songID', 'rating']

Названия колонок после обработки: ['userID', 'songID', 'rating']

Первые 5 строк данных:
  userID songID rating
0      0   7171      5
1      0   8637      4
2      0  21966      4
3      0  35821      5
4      0  82446      5

Информация о данных:
<class 'pandas.DataFrame'>
RangeIndex: 49999 entries, 0 to 49998
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   userID  49999 non-null  str  
 1   songID  49999 non-null  str  
 2   rating  49999 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB
None

Статистика по рейтингам:
count    49999.000000
mean         3.467009
std          1.547295
min          1.000000
25%          2.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: rating, dtype: float64

Количество уника

In [65]:
# ------------------------------
# 2. Очистка данных и преобразование
# ------------------------------


# Проверяем, что рейтинги находятся в разумных пределах (например, 1-5)
if not ((df['rating'] >= 1) & (df['rating'] <= 5)).all():
    print("Обнаружены рейтинги вне диапазона 1-5. Удаляем такие строки.")
    df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Удаляем явные дубликаты (если один пользователь поставил одну и ту же песню несколько раз)
duplicates = df.duplicated(subset=['userID', 'songID']).sum()
if duplicates > 0:
    print(f"Найдено {duplicates} дубликатов. Удаляем их, оставляя первое вхождение.")
    df = df.drop_duplicates(subset=['userID', 'songID'], keep='first')

# Сбрасываем индексы после удалений
df = df.reset_index(drop=True)

# Преобразуем идентификаторы в индексы, начинающиеся с 0 (для эмбеддингов PyTorch)
# Сначала создадим словари для отображения исходных ID в последовательные индексы
unique_users = df['userID'].unique()
unique_items = df['songID'].unique()

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
item_to_idx = {iid: i for i, iid in enumerate(unique_items)}

# Добавляем столбцы с индексами
df['user_idx'] = df['userID'].map(user_to_idx)
df['item_idx'] = df['songID'].map(item_to_idx)

# Теперь можно использовать user_idx и item_idx
num_users = len(unique_users)
num_items = len(unique_items)

print(f"\nПосле очистки: пользователей = {num_users}, объектов = {num_items}, взаимодействий = {len(df)}")


После очистки: пользователей = 5000, объектов = 24752, взаимодействий = 49999


In [66]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

# Создаём полный датасет
dataset = RatingsDataset(df)

# Разделяем на train (80%) и test (20%)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:


class RecommenderNN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super(RecommenderNN, self).__init__()
        # Эмбеддинги пользователей и объектов
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # Полносвязные слои для предсказания рейтинга
        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, item):
        user_emb = self.user_embedding(user)
        item_emb = self.item_embedding(item)
        x = torch.cat([user_emb, item_emb], dim=1)
        return self.fc_layers(x).squeeze()

model = RecommenderNN(num_users, num_items, embedding_dim=32).to(device)
print(model)



RecommenderNN(
  (user_embedding): Embedding(5000, 32)
  (item_embedding): Embedding(24752, 32)
  (fc_layers): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
epochs = 10

train_losses = []
train_rmses = []
train_maes = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    all_preds = []
    all_true = []

    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)

        optimizer.zero_grad()
        predictions = model(users, items)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(predictions.cpu().detach().numpy())
        all_true.extend(ratings.cpu().detach().numpy())

    avg_loss = total_loss / len(train_loader)
    rmse = math.sqrt(mean_squared_error(all_true, all_preds))
    mae = mean_absolute_error(all_true, all_preds)

    train_losses.append(avg_loss)
    train_rmses.append(rmse)
    train_maes.append(mae)

    print(f'Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}')

Epoch  1 | Loss: 2.5670 | RMSE: 1.6022 | MAE: 1.3786
Epoch  2 | Loss: 1.8076 | RMSE: 1.3445 | MAE: 1.1216
Epoch  3 | Loss: 1.3533 | RMSE: 1.1633 | MAE: 0.9363
Epoch  4 | Loss: 0.9507 | RMSE: 0.9750 | MAE: 0.7622
Epoch  5 | Loss: 0.6281 | RMSE: 0.7925 | MAE: 0.6027
Epoch  6 | Loss: 0.4135 | RMSE: 0.6430 | MAE: 0.4834
Epoch  7 | Loss: 0.2785 | RMSE: 0.5277 | MAE: 0.3964
Epoch  8 | Loss: 0.2021 | RMSE: 0.4496 | MAE: 0.3410
Epoch  9 | Loss: 0.1613 | RMSE: 0.4017 | MAE: 0.3099
Epoch 10 | Loss: 0.1374 | RMSE: 0.3707 | MAE: 0.2900


In [ ]:
model.eval()
test_preds = []
test_true = []

with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        predictions = model(users, items)
        test_preds.extend(predictions.cpu().numpy())
        test_true.extend(ratings.cpu().numpy())

test_rmse = math.sqrt(mean_squared_error(test_true, test_preds))
test_mae = mean_absolute_error(test_true, test_preds)

print(f'\nТестовые метрики: RMSE = {test_rmse:.4f}, MAE = {test_mae:.4f}')


def recommend_for_user(original_user_id, model, user_to_idx, item_to_idx, df, n_recommendations=10):
    """
    Возвращает топ-N объектов для пользователя с исходным ID, исключая уже оценённые.
    """
    if original_user_id not in user_to_idx:
        print(f"Пользователь {original_user_id} не найден.")
        return []

    user_idx = user_to_idx[original_user_id]

    rated_items = set(df[df['user_idx'] == user_idx]['item_idx'].values)

    all_items = torch.tensor(list(range(num_items)), dtype=torch.long).to(device)
    user_tensor = torch.full((num_items,), user_idx, dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        predictions = model(user_tensor, all_items).cpu().numpy()

    item_indices = np.argsort(predictions)[::-1]

    recommendations = []
    for idx in item_indices:
        if idx not in rated_items:
            recommendations.append(idx)
            if len(recommendations) == n_recommendations:
                break

    idx_to_item = {v: k for k, v in item_to_idx.items()}
    recommended_original_ids = [idx_to_item[i] for i in recommendations]

    return recommended_original_ids

example_user = unique_users[0]
print(f"\nРекомендации для пользователя {example_user}:")
recs = recommend_for_user(example_user, model, user_to_idx, item_to_idx, df, n_recommendations=5)
for i, item_id in enumerate(recs, 1):
    print(f"{i}. Объект {item_id}")



Тестовые метрики: RMSE = 1.6234, MAE = 1.3098

Рекомендации для пользователя 0:
1. Объект 131129
2. Объект 43248
3. Объект 75870
4. Объект 83306
5. Объект 51045
